# 3. Suggestion de modes de défaillance (AMDEC)

Pour un élément et une fonction, le modèle propose des modes de défaillance à partir de mots-guides
(perte, intempestif, dégradé, intermittent, retardé, figé, valeur erronée), avec effet, cause et cotations S/O/D.

Les modes déjà présents dans votre AMDEC sont envoyés au modèle pour éviter les doublons.

> ⚠️ Les cotations proposées sont des **estimations à valider** en groupe de travail AMDEC.

In [1]:
# Rend le package safety_agent importable depuis le dossier notebooks/
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
EXAMPLES = ROOT / "examples"

import pandas as pd
pd.set_option("display.max_colwidth", 120)

In [2]:
# Configuration du modèle hébergé (Ollama par défaut)
BASE_URL = "http://localhost:11434/v1"   # vLLM : "http://serveur:8000/v1"
MODEL = "qwen2.5:14b"

from safety_agent.notebook import get_llm
llm = get_llm(BASE_URL, MODEL)   # bascule en MODE DÉMO si le serveur ne répond pas

⚠️  Serveur LLM injoignable à http://localhost:11434/v1 (<urlopen error [Errno 111] Connection refused>)
➡️  MODE DÉMO : réponses simulées, sans IA. Lancez Ollama/vLLM pour de vraies analyses.


In [3]:
from safety_agent.loaders import load_project

# Remplacez par vos fichiers (.csv ou .xlsx)
REQS = EXAMPLES / "bms_exigences.csv"
FMEA = EXAMPLES / "bms_amdec.csv"

project, load_findings = load_project(REQS, FMEA)
print(f"{len(project.requirements)} exigences, {len(project.fmea)} lignes AMDEC")

11 exigences, 6 lignes AMDEC


In [4]:
ITEM = "Capteur de courant pack"
FUNCTION = "Mesurer le courant de charge et de décharge de la batterie"
CONTEXT = "BMS 400 V de véhicule électrique, capteur à effet Hall, objectif SG-01 ASIL C"

In [5]:
from safety_agent.ai import suggest_failure_modes
from safety_agent.checks.action_priority import action_priority

proposals = suggest_failure_modes(llm, project, ITEM, FUNCTION, CONTEXT)
rows = []
for p in proposals:
    d = p.data
    try:
        ap = action_priority(int(d["S"]), int(d["O"]), int(d["D"]))
    except (KeyError, ValueError, TypeError):
        ap = "?"
    rows.append({"Mode": d.get("mode"), "Mot-guide": d.get("mot_guide"), "Effet": d.get("effet"),
                 "Cause": d.get("cause"), "S": d.get("S"), "O": d.get("O"), "D": d.get("D"), "AP": ap,
                 "Détection": d.get("detection")})
pd.DataFrame(rows)

,Mode,Mot-guide,Effet,Cause,S,O,D,AP,Détection
0,[DÉMO] Perte du signal (Capteur de courant pack),perte de fonction,Information indisponible pour la fonction de sécurité,"Rupture de connexion, défaut d'alimentation",8,3,3,L,Diagnostic de circuit ouvert
1,[DÉMO] Valeur figée (Capteur de courant pack),fonction figée,Dérive non détectée de la grandeur surveillée,"Blocage du convertisseur, défaut logiciel",9,3,5,M,Contrôle de variation / plausibilité
2,[DÉMO] Valeur erronée hors tolérance (Capteur de courant pack),valeur erronée,Décision de sécurité prise sur une mesure fausse,"Dérive en température, vieillissement",9,4,5,H,Comparaison avec une mesure redondante


## Exporter au format AMDEC (CSV, ouvrable dans Excel)

In [6]:
from safety_agent.report import write_proposals_csv

out = ROOT / "propositions_amdec.csv"
write_proposals_csv(proposals, out)
print("Export :", out.relative_to(ROOT))

Export : propositions_amdec.csv
